# Módulo EP2 — Planificación, Herramientas, Orquestación y Memoria

Correspondencia directa con:
- `app/agentic/planner.py` (`RuleBasedPlanner`)
- `app/agentic/orchestrator.py` (`AdaptiveOrchestrator`)
- `app/agentic/service.py` (`AgentService`)
- `app/tools/knowledge.py` y `app/tools/incidents.py`
- `app/memory/short_term.py` y `app/memory/write_back.py`


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HikariLucy/KnowledgeFlow-AgentOps.git"
REPO_BRANCH = "feat/eft-notebook"

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "app" / "main.py").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None and IN_COLAB:
    target = Path("/content/KnowledgeFlow-AgentOps")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(target)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(target / "requirements.txt")], check=True)
    repo_root = target

if repo_root is None:
    raise RuntimeError("No se encontró la raíz de KnowledgeFlow-AgentOps. Ejecuta desde el repositorio o en Google Colab.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Seguridad: modo offline sin claves de Gemini por defecto
os.environ.setdefault("OBS_EXECUTION_MODE", "offline")
os.environ.pop("GEMINI_API_KEY", None)
RUN_LIVE_GEMINI = False

print(f"Entorno listo | IN_COLAB={IN_COLAB} | repo_root={repo_root} | RUN_LIVE_GEMINI={RUN_LIVE_GEMINI}")


## 1. Inspección fiel del código fuente EP2 (sin duplicación manual)


In [ ]:
import inspect
from app.agentic.service import AgentService
from app.memory.short_term import ShortTermMemory

print("=== app/agentic/service.py :: AgentService.execute ===")
print(inspect.getsource(AgentService.execute))
print("=== app/memory/short_term.py :: ShortTermMemory.add_message ===")
print(inspect.getsource(ShortTermMemory.add_message))


## 2. Ejecución offline de herramientas y continuidad conversacional en SQLite temporal


In [ ]:
from tempfile import TemporaryDirectory
from app.storage.database import SQLiteDatabase
from app.storage.repositories import IncidentRepository
from app.memory.short_term import ShortTermMemory
from app.memory.write_back import MemoryWriteBack
from app.agentic.planner import RuleBasedPlanner
from app.agentic.orchestrator import AdaptiveOrchestrator
from app.agentic.service import AgentService
from app.api.agent_schemas import AgentRequest
from app.tools.incidents import CreateIncidentTool, SearchIncidentsTool, AppendIncidentNoteTool
from app.tools.schemas import CreateIncidentInput, SearchIncidentsInput, AppendIncidentNoteDraft

with TemporaryDirectory() as tmp_dir:
    db = SQLiteDatabase(Path(tmp_dir) / "ep2_incidents.sqlite3")
    db.initialize()
    repo = IncidentRepository(db)
    stm = ShortTermMemory(max_turns=4)
    wb = MemoryWriteBack(short_term_memory=stm)
    service = AgentService(
        AdaptiveOrchestrator(
            planner=RuleBasedPlanner(short_term_memory=stm),
            create_incident_tool=CreateIncidentTool(repository=repo),
            search_incidents_tool=SearchIncidentsTool(repository=repo),
            append_incident_note_tool=AppendIncidentNoteTool(repository=repo),
            memory_write_back=wb,
        )
    )
    t1 = service.execute(AgentRequest(
        conversation_id="ep2-conv",
        message="Crear incidente por caída de servicio",
        create_incident=CreateIncidentInput(
            title="Caída de portal interno",
            description="Error 503 intermitente en portal de soporte.",
            category="software",
            severity="medium",
        ),
    ))
    t2 = service.execute(AgentRequest(
        conversation_id="ep2-conv",
        message="Buscar incidentes de portal",
        search_incidents=SearchIncidentsInput(query="portal"),
    ))
    t3 = service.execute(AgentRequest(
        conversation_id="ep2-conv",
        message="Agregar nota de diagnóstico al incidente",
        append_incident_note=AppendIncidentNoteDraft(note="Reiniciado balanceador en ambiente local."),
    ))
    print("Turno 1:", t1.status, t1.incident_id, t1.tool_calls)
    print("Turno 2:", t2.status, t2.tool_calls)
    print("Turno 3:", t3.status, t3.incident_id, "mensajes en memoria:", len(stm.get_messages("ep2-conv")))
